In [ ]:
import zipfile
import shutil

# Unzip the adapter
with zipfile.ZipFile('/content/lfm2_finetuned (1).zip', 'r') as zip_ref:
    zip_ref.extractall('/content/lfm2_finetuned')

print("✅ Adapter extracted!")

✅ Adapter extracted!


In [ ]:
from unsloth import FastModel
import torch

# Load base model
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/LFM2-1.2B",
    dtype=None,
    max_seq_length=2048,
    load_in_4bit=False,
)

# Load your trained adapter
from peft import PeftModel

model = PeftModel.from_pretrained(model, "/content/lfm2_finetuned")

print("✅ Adapter loaded!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/209 [00:00<?, ?B/s]

✅ Adapter loaded!


In [ ]:
model = model.merge_and_unload()

In [ ]:
# Test inference with the merged model
import json
import re

def clean_and_format_recipe(raw_output):
    """
    Cleans hallucinated/malformed recipe JSON and formats it nicely.
    Handles typos, empty steps, and malformed JSON.
    """
    try:
        # Try to extract JSON from the output
        # Sometimes model adds extra text before/after JSON
        json_match = re.search(r'\{.*\}', raw_output, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
        else:
            json_str = raw_output

        # Parse JSON (handles common typos)
        data = json.loads(json_str)

        # Handle common typos in keys
        ingredients = None
        directions = None

        # Check for ingredients (with typo variations)
        for key in ['ingredients', 'ingrediens', 'ingredient', 'ingridients','ingrediences']:
            if key in data:
                ingredients = data[key]
                break

        # Check for directions (with typo variations)
        for key in ['directions', 'direction', 'steps', 'instructions']:
            if key in data:
                directions = data[key]
                break

        if not ingredients:
            ingredients = []
        if not directions:
            directions = []

        # Clean ingredients
        cleaned_ingredients = []
        for ing in ingredients:
            if isinstance(ing, str):
                ing = ing.strip()
                # Remove empty strings and just dashes
                if ing and ing not in ['-', '—', '•', '*', '']:
                    cleaned_ingredients.append(ing)

        # Clean directions
        cleaned_directions = []
        for direction in directions:
            if isinstance(direction, str):
                direction = direction.strip()
                # Remove leading bullets/dashes
                direction = re.sub(r'^[-•*.\s]+', '', direction)
                # Remove empty strings and just punctuation
                if direction and len(direction) > 2:
                    cleaned_directions.append(direction)

        # Format output
        output = []
        output.append("=" * 60)
        output.append("RECIPE")
        output.append("=" * 60)

        # Ingredients section
        output.append("\nIngredients:")
        output.append("-" * 60)
        if cleaned_ingredients:
            for ing in cleaned_ingredients:
                output.append(f"  • {ing}")
        else:
            output.append("  (No ingredients found)")

        # Directions section
        output.append("\nDirections:")
        output.append("-" * 60)
        if cleaned_directions:
            for i, direction in enumerate(cleaned_directions, 1):
                output.append(f"  {i}. {direction}")
        else:
            output.append("  (No directions found)")

        output.append("\n" + "=" * 60)

        return "\n".join(output), cleaned_ingredients, cleaned_directions

    except json.JSONDecodeError as e:
        return f"❌ Failed to parse JSON: {e}\n\nRaw output:\n{raw_output}", [], []
    except Exception as e:
        return f"❌ Error processing recipe: {e}\n\nRaw output:\n{raw_output}", [], []

print("Testing inference...")

# Exact same prompt format as training
prompt = "Output the ingredients and directions for making Sushi in JSON format with two keys: 'ingredients' and 'directions'."

# Apply chat template
input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)

# Generate with proper anti-repetition parameters
output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.05,
    no_repeat_ngram_size=3,
    max_new_tokens=1024,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

response = tokenizer.decode(output[0], skip_special_tokens=True)
assistant_response = response.split("assistant")[-1].strip()
formatted_recipe, ingredients, directions = clean_and_format_recipe(assistant_response)
print(formatted_recipe)

# Decode and print
#response = tokenizer.decode(output[0], skip_special_tokens=True)
# print("\n" + "="*80)
# print("FULL OUTPUT:")
# print("="*80)
# print(response)
# print("="*80)
# formatted_output, ingredients, directions = clean_and_format_recipe(response)
# print(formatted_output)


# Extract just the assistant's response
# assistant_response = response.split("assistant")[-1].strip()
# print("\nASSISTANT RESPONSE ONLY:")
# print("="*80)
# print(assistant_response)
# print("="*80)


Testing inference...
RECIPE

Ingredients:
------------------------------------------------------------
  • Sushi Rice
  • 1 cup white rice
  • 2 cups water
  • Salt to taste
  • Sushi Vegetables
  • 6 ounces cucumber, peeled and sliced
  • 4 ounces carrot, peel and sliced into thin strips
  • 3 ounces radish, peel, cut into thin slices
  • 5 ounces green beans, trimmed and cut into 1-inch chunks
  • Fish
  • 7 ounces cooked salmon or other firm white fish, cubed
  • Green Onion
  • 10 ounces thinly sliced green onions

Directions:
------------------------------------------------------------
  1. Place the rice, water and salt in a medium saucepan. Bring to a boil, then reduce heat to low and simmer for 20 minutes. Remove from heat and allow to stand, covered, for 5 minutes.
  2. Stir the vegetables into the rice. Cover and let stand for another 10 minutes.
  3. In a medium bowl, mix together the vegetable mixture, fish and green onions. Divide evenly among 6 sushi rolls and fill each w